# nb2: Local Hugging Face-Only Category Pipeline (M1 Mac)

This notebook builds a **local classification pipeline** that assigns each input to one of your predefined categories.

## What it supports
- Inline text (`classify_text_payload`)
- Files (`classify_file`) including text, PDF, DOCX, CSV/TSV/XLSX, JSON, and images
- Batch inputs (`classify_many`)

## Hugging Face models used (up to 4)
1. `BAAI/bge-m3` (semantic embeddings)
2. `papluca/xlm-roberta-base-language-detection` (language detection)
3. `BAAI/bge-reranker-v2-m3` (optional disambiguation reranker)
4. `google/siglip2-base-patch16-224` (optional image classification via prompt similarity)

This is designed for local execution on an M1 Mac and can run fully offline if models are cached.


In [1]:
# Run once if needed (uncomment):
# %pip install -q torch torchvision torchaudio transformers sentence-transformers pillow pandas numpy PyPDF2 python-docx openpyxl filetype tqdm


In [2]:
from __future__ import annotations

import json
import mimetypes
import os
import time
from pathlib import Path
from typing import Any, Dict, List, Tuple, Union

import numpy as np
import pandas as pd
import torch
from PIL import Image
import PyPDF2
from transformers import AutoModel, AutoModelForSequenceClassification, AutoProcessor, AutoTokenizer
from sentence_transformers import SentenceTransformer

try:
    from docx import Document
    HAS_DOCX = True
except Exception:
    Document = None
    HAS_DOCX = False

try:
    import filetype
    HAS_FILETYPE = True
except Exception:
    filetype = None
    HAS_FILETYPE = False

BASE_DIR = Path('/Users/benjaminfalkenburg/Documents/Shannova/Beta')
CACHE_DIR = BASE_DIR / 'cache' / 'hf_models'
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OFFLINE_MODE = True  # Set to False if you want model downloads when missing
if OFFLINE_MODE:
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    os.environ['HF_DATASETS_OFFLINE'] = '1'

if CACHE_DIR.exists():
    os.environ.setdefault('HUGGINGFACE_HUB_CACHE', str(CACHE_DIR))
    os.environ.setdefault('TRANSFORMERS_CACHE', str(CACHE_DIR))

MPS_AVAILABLE = bool(getattr(torch.backends, 'mps', None)) and torch.backends.mps.is_available()
DEFAULT_DEVICE = 'mps' if MPS_AVAILABLE else 'cpu'

print('BASE_DIR:', BASE_DIR)
print('CACHE_DIR:', CACHE_DIR)
print('OFFLINE_MODE:', OFFLINE_MODE)
print('DEFAULT_DEVICE:', DEFAULT_DEVICE)
print('HAS_DOCX:', HAS_DOCX)
print('HAS_FILETYPE:', HAS_FILETYPE)


BASE_DIR: /Users/benjaminfalkenburg/Documents/Shannova/Beta
CACHE_DIR: /Users/benjaminfalkenburg/Documents/Shannova/Beta/cache/hf_models
OFFLINE_MODE: True
DEFAULT_DEVICE: mps
HAS_DOCX: True
HAS_FILETYPE: True


In [3]:
# Predefined categories (edit these definitions to match your business taxonomy)
CATEGORY_DEFINITIONS: Dict[str, Dict[str, Any]] = {
    'FINANCIAL': {
        'description': 'Accounting, banking, investments, transactions, valuation, pricing, and market data.',
        'keywords': ['revenue', 'profit', 'cash flow', 'balance sheet', 'invoice', 'portfolio', 'stock', 'payment', 'budget', 'forecast'],
        'examples': ['quarterly earnings statement', 'bank transaction log', 'investment portfolio report'],
        'image_prompt': 'A financial document, spreadsheet, invoice, bank statement, trading chart, or accounting dashboard.'
    },
    'HEALTH_MEDICAL': {
        'description': 'Clinical care, health records, diagnostics, treatment outcomes, epidemiology, and biomedical data.',
        'keywords': ['patient', 'diagnosis', 'treatment', 'clinical', 'hospital', 'symptom', 'trial', 'biomarker', 'prescription', 'disease'],
        'examples': ['patient discharge summary', 'clinical trial dataset', 'hospital quality metrics'],
        'image_prompt': 'A medical chart, lab report, hospital scene, diagnostic image, or healthcare dashboard.'
    },
    'SCIENCE_RESEARCH': {
        'description': 'Scientific experiments, hypotheses, measurements, reproducibility, and research outputs.',
        'keywords': ['experiment', 'methodology', 'hypothesis', 'analysis', 'dataset', 'publication', 'laboratory', 'observation', 'replicate', 'study'],
        'examples': ['research abstract', 'lab experiment results', 'scientific benchmark table'],
        'image_prompt': 'A scientific chart, microscope scene, research paper figure, or laboratory setup.'
    },
    'TECHNOLOGY_IT': {
        'description': 'Software, APIs, cloud systems, cybersecurity, logs, source code, and technical architecture.',
        'keywords': ['api', 'source code', 'deployment', 'cloud', 'container', 'database', 'latency', 'backend', 'frontend', 'infrastructure'],
        'examples': ['service log file', 'software architecture doc', 'application source code'],
        'image_prompt': 'A software dashboard, code editor, cloud architecture diagram, or IT operations screen.'
    },
    'BUSINESS_COMMERCIAL': {
        'description': 'Operations, sales, customer behavior, product performance, and commercial planning.',
        'keywords': ['customer', 'sales', 'conversion', 'retention', 'churn', 'pricing strategy', 'market segment', 'pipeline', 'kpi', 'operations'],
        'examples': ['sales pipeline report', 'customer segmentation dataset', 'operational KPI review'],
        'image_prompt': 'A business analytics dashboard, sales chart, operational report, or corporate presentation slide.'
    },
    'LEGAL_GOVERNMENT': {
        'description': 'Legal text, policy, regulation, compliance, contracts, and public-sector documents.',
        'keywords': ['regulation', 'statute', 'contract', 'compliance', 'clause', 'policy', 'court', 'government', 'law', 'procurement'],
        'examples': ['service contract', 'regulatory compliance report', 'government policy memo'],
        'image_prompt': 'A legal document, policy brief, government form, courtroom visual, or regulation checklist.'
    },
    'EDUCATION_ACADEMIC': {
        'description': 'Learning materials, curriculum, teaching outcomes, and educational assessment data.',
        'keywords': ['curriculum', 'syllabus', 'student', 'assessment', 'grade', 'course', 'lecture', 'pedagogy', 'learning', 'classroom'],
        'examples': ['course syllabus', 'student performance table', 'educational outcomes report'],
        'image_prompt': 'A classroom scene, textbook page, learning dashboard, or school performance chart.'
    },
    'CREATIVE_MEDIA': {
        'description': 'Content creation, media assets, audience engagement, storytelling, and production workflows.',
        'keywords': ['video', 'audio', 'campaign', 'creative', 'content', 'engagement', 'editorial', 'script', 'brand', 'social media'],
        'examples': ['youtube analytics export', 'campaign performance summary', 'editorial content plan'],
        'image_prompt': 'A media production workspace, camera shot, content dashboard, or social engagement chart.'
    },
    'PERSONAL_INFORMAL': {
        'description': 'Personal notes, casual messages, informal writing, and non-professional records.',
        'keywords': ['personal', 'note', 'chat', 'message', 'diary', 'reminder', 'to-do', 'informal', 'family', 'daily'],
        'examples': ['personal journal entry', 'informal chat transcript', 'daily to-do list'],
        'image_prompt': 'A personal note, casual memo, chat screenshot style, or everyday informal document.'
    },
    'ENGINEERING_MANUFACTURING': {
        'description': 'Industrial processes, mechanical systems, quality control, and production operations.',
        'keywords': ['manufacturing', 'assembly', 'tolerance', 'specification', 'machine', 'defect', 'quality control', 'plant', 'maintenance', 'throughput'],
        'examples': ['factory quality report', 'machine maintenance log', 'production line metrics'],
        'image_prompt': 'An engineering drawing, factory floor, machinery dashboard, or manufacturing quality chart.'
    },
    'ENVIRONMENTAL_SUSTAINABILITY': {
        'description': 'Climate, emissions, energy use, biodiversity, environmental impact, and sustainability tracking.',
        'keywords': ['emissions', 'climate', 'carbon', 'sustainability', 'energy', 'renewable', 'waste', 'water', 'ecosystem', 'environmental'],
        'examples': ['carbon emissions inventory', 'renewable energy report', 'environmental impact dataset'],
        'image_prompt': 'An environmental monitoring dashboard, climate graph, renewable energy scene, or sustainability report.'
    },
    'GENERAL_OTHER': {
        'description': 'Catch-all class for content that does not strongly fit any specialized category.',
        'keywords': ['miscellaneous', 'general', 'other', 'unclear', 'mixed'],
        'examples': ['mixed content document', 'generic data extract', 'uncategorized note'],
        'image_prompt': 'A generic document, mixed-content screenshot, or uncategorized visual material.'
    },
}

CATEGORY_LABELS: List[str] = list(CATEGORY_DEFINITIONS.keys())
print('Categories:', CATEGORY_LABELS)


Categories: ['FINANCIAL', 'HEALTH_MEDICAL', 'SCIENCE_RESEARCH', 'TECHNOLOGY_IT', 'BUSINESS_COMMERCIAL', 'LEGAL_GOVERNMENT', 'EDUCATION_ACADEMIC', 'CREATIVE_MEDIA', 'PERSONAL_INFORMAL', 'ENGINEERING_MANUFACTURING', 'ENVIRONMENTAL_SUSTAINABILITY', 'GENERAL_OTHER']


In [4]:
MODEL_IDS = {
    'embed_text': 'BAAI/bge-m3',
    'lang_id': 'papluca/xlm-roberta-base-language-detection',
    'reranker': 'BAAI/bge-reranker-v2-m3',
    'image_vl': 'google/siglip2-base-patch16-224',
}

USE_RERANKER = True
USE_IMAGE_MODEL = True

MAX_TEXT_CHARS = 12000
MAX_TEXT_CHUNKS = 8
CHUNK_SIZE = 1400
CHUNK_OVERLAP = 250
TOP_K_DEFAULT = 3

TEXT_TEMPERATURE = 0.08
RERANKER_BLEND = 0.25


def _load_seq_model(model_id: str, preferred_device: str):
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        cache_dir=str(CACHE_DIR),
        local_files_only=OFFLINE_MODE,
    )
    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            model_id,
            cache_dir=str(CACHE_DIR),
            local_files_only=OFFLINE_MODE,
        )
        model = model.to(preferred_device)
        device = preferred_device
    except Exception as e:
        print(f'[warn] {model_id} failed on {preferred_device}; using cpu. Reason: {e}')
        model = AutoModelForSequenceClassification.from_pretrained(
            model_id,
            cache_dir=str(CACHE_DIR),
            local_files_only=OFFLINE_MODE,
        )
        model = model.to('cpu')
        device = 'cpu'
    model.eval()
    return tokenizer, model, device


def _load_embed_model(model_id: str, preferred_device: str):
    try:
        model = SentenceTransformer(
            model_id,
            cache_folder=str(CACHE_DIR),
            device=preferred_device,
            trust_remote_code=True,
        )
        return model, preferred_device
    except Exception as e:
        print(f'[warn] {model_id} failed on {preferred_device}; using cpu. Reason: {e}')
        model = SentenceTransformer(
            model_id,
            cache_folder=str(CACHE_DIR),
            device='cpu',
            trust_remote_code=True,
        )
        return model, 'cpu'


embed_model, embed_device = _load_embed_model(MODEL_IDS['embed_text'], DEFAULT_DEVICE)
lang_tokenizer, lang_model, lang_device = _load_seq_model(MODEL_IDS['lang_id'], DEFAULT_DEVICE)

reranker_tokenizer = None
reranker_model = None
reranker_device = 'cpu'
if USE_RERANKER:
    try:
        reranker_tokenizer, reranker_model, reranker_device = _load_seq_model(MODEL_IDS['reranker'], DEFAULT_DEVICE)
    except Exception as e:
        print('[warn] Reranker disabled:', e)
        USE_RERANKER = False

image_processor = None
image_model = None
image_device = 'cpu'
if USE_IMAGE_MODEL:
    try:
        image_processor = AutoProcessor.from_pretrained(
            MODEL_IDS['image_vl'],
            cache_dir=str(CACHE_DIR),
            local_files_only=OFFLINE_MODE,
        )
        image_model = AutoModel.from_pretrained(
            MODEL_IDS['image_vl'],
            cache_dir=str(CACHE_DIR),
            local_files_only=OFFLINE_MODE,
        )
        try:
            image_model = image_model.to(DEFAULT_DEVICE)
            image_device = DEFAULT_DEVICE
        except Exception:
            image_model = image_model.to('cpu')
            image_device = 'cpu'
        image_model.eval()
    except Exception as e:
        print('[warn] Image model disabled:', e)
        USE_IMAGE_MODEL = False


def _category_profile_text(label: str, cfg: Dict[str, Any]) -> str:
    keywords = ', '.join(cfg['keywords'])
    examples = ' | '.join(cfg['examples'])
    return f"Category: {label}. Description: {cfg['description']} Keywords: {keywords}. Examples: {examples}."


CATEGORY_PROFILES: Dict[str, str] = {
    label: _category_profile_text(label, cfg)
    for label, cfg in CATEGORY_DEFINITIONS.items()
}

CATEGORY_IMAGE_PROMPTS: List[str] = [
    CATEGORY_DEFINITIONS[label]['image_prompt'] for label in CATEGORY_LABELS
]

# Build category prototype vectors from multiple prompts per category
category_vectors: List[np.ndarray] = []
for label in CATEGORY_LABELS:
    cfg = CATEGORY_DEFINITIONS[label]
    profile_texts = [
        CATEGORY_PROFILES[label],
        f"{label}: {cfg['description']}",
        f"Keywords: {', '.join(cfg['keywords'])}",
    ] + cfg['examples']

    embs = embed_model.encode(
        profile_texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        batch_size=8,
    )
    embs = np.asarray(embs, dtype=np.float32)
    proto = embs.mean(axis=0)
    proto = proto / (np.linalg.norm(proto) + 1e-12)
    category_vectors.append(proto.astype(np.float32))

CATEGORY_PROTO_MATRIX = np.vstack(category_vectors)

print('Embedding model:', MODEL_IDS['embed_text'], '| device:', embed_device)
print('Language model:', MODEL_IDS['lang_id'], '| device:', lang_device)
print('Reranker enabled:', USE_RERANKER, '| device:', reranker_device)
print('Image model enabled:', USE_IMAGE_MODEL, '| device:', image_device)
print('Category prototypes shape:', CATEGORY_PROTO_MATRIX.shape)


Python(90820) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Embedding model: BAAI/bge-m3 | device: mps
Language model: papluca/xlm-roberta-base-language-detection | device: mps
Reranker enabled: True | device: mps
Image model enabled: True | device: mps
Category prototypes shape: (12, 1024)


In [5]:
def _softmax(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x)
    ex = np.exp(x)
    denom = ex.sum()
    if denom <= 0:
        return np.ones_like(ex) / len(ex)
    return ex / denom


def _chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    text = (text or '').strip()
    if not text:
        return []
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    step = max(1, chunk_size - overlap)
    for i in range(0, len(text), step):
        chunk = text[i:i + chunk_size]
        if chunk:
            chunks.append(chunk)
        if i + chunk_size >= len(text):
            break
    return chunks


def detect_language(text: str, max_chars: int = 2000) -> Tuple[str, float]:
    sample = (text or '').strip()[:max_chars]
    if not sample:
        return 'unknown', 0.0

    with torch.no_grad():
        inputs = lang_tokenizer(sample, return_tensors='pt', truncation=True, max_length=512)
        inputs = {k: v.to(lang_device) for k, v in inputs.items()}
        logits = lang_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]
        idx = int(torch.argmax(probs).item())
        conf = float(probs[idx].item())

    label = lang_model.config.id2label.get(idx, str(idx))
    return label, conf


def _keyword_boost(text: str) -> np.ndarray:
    text_low = (text or '').lower()
    boost = np.zeros(len(CATEGORY_LABELS), dtype=np.float32)

    for idx, label in enumerate(CATEGORY_LABELS):
        keywords = CATEGORY_DEFINITIONS[label]['keywords']
        hits = sum(1 for kw in keywords if kw.lower() in text_low)
        boost[idx] = min(0.02 * hits, 0.16)

    return boost


def _semantic_scores(text: str) -> np.ndarray:
    chunks = _chunk_text(text[:MAX_TEXT_CHARS])[:MAX_TEXT_CHUNKS]
    if not chunks:
        return np.zeros(len(CATEGORY_LABELS), dtype=np.float32)

    chunk_embs = embed_model.encode(
        chunks,
        convert_to_numpy=True,
        normalize_embeddings=True,
        batch_size=8,
    )
    chunk_embs = np.asarray(chunk_embs, dtype=np.float32)

    doc_vec = chunk_embs.mean(axis=0)
    doc_vec = doc_vec / (np.linalg.norm(doc_vec) + 1e-12)

    doc_scores = CATEGORY_PROTO_MATRIX @ doc_vec
    chunk_scores = chunk_embs @ CATEGORY_PROTO_MATRIX.T
    max_chunk_scores = chunk_scores.max(axis=0)

    return (0.65 * doc_scores + 0.35 * max_chunk_scores).astype(np.float32)


def _rerank_scores(text: str, candidate_labels: List[str]) -> Dict[str, float]:
    if (not USE_RERANKER) or (reranker_model is None):
        return {label: 0.0 for label in candidate_labels}

    short_text = (text or '')[:2000]
    left = [short_text] * len(candidate_labels)
    right = [CATEGORY_PROFILES[label] for label in candidate_labels]

    inputs = reranker_tokenizer(
        left,
        right,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors='pt',
    )
    inputs = {k: v.to(reranker_device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = reranker_model(**inputs).logits.squeeze(-1)

    arr = logits.detach().float().cpu().numpy()
    return {label: float(arr[i]) for i, label in enumerate(candidate_labels)}


def classify_text_payload(
    text: str,
    source: str = 'inline_text',
    top_k: int = TOP_K_DEFAULT,
    use_reranker: bool = USE_RERANKER,
) -> Dict[str, Any]:
    start = time.time()

    raw = '' if text is None else str(text)
    cleaned = ' '.join(raw.replace('\x00', ' ').split())
    cleaned = cleaned[:MAX_TEXT_CHARS]

    if not cleaned:
        return {
            'source': source,
            'input_type': 'text',
            'content_category': 'GENERAL_OTHER',
            'confidence': 0.0,
            'top_categories': [],
            'language': 'unknown',
            'language_confidence': 0.0,
            'status': 'error',
            'error': 'Empty text payload',
            'processing_time_ms': round((time.time() - start) * 1000, 2),
        }

    language, language_conf = detect_language(cleaned)

    base_scores = _semantic_scores(cleaned) + _keyword_boost(cleaned)
    probs = _softmax(base_scores / TEXT_TEMPERATURE)

    if use_reranker and USE_RERANKER and reranker_model is not None:
        cand_idx = np.argsort(probs)[::-1][:min(4, len(CATEGORY_LABELS))]
        cand_labels = [CATEGORY_LABELS[i] for i in cand_idx]
        rr = _rerank_scores(cleaned, cand_labels)
        rr_logits = np.array([rr[label] for label in cand_labels], dtype=np.float32)
        rr_probs = _softmax(rr_logits)

        for i, idx in enumerate(cand_idx):
            probs[idx] = (1 - RERANKER_BLEND) * probs[idx] + RERANKER_BLEND * rr_probs[i]
        probs = probs / probs.sum()

    order = np.argsort(probs)[::-1]
    top_idx = order[:max(1, top_k)]

    top_categories = [
        {
            'category': CATEGORY_LABELS[i],
            'probability': float(probs[i]),
        }
        for i in top_idx
    ]

    best = top_categories[0]
    second_prob = top_categories[1]['probability'] if len(top_categories) > 1 else 0.0
    margin = best['probability'] - second_prob

    confidence = float(min(1.0, best['probability'] + 0.25 * margin))
    if len(cleaned) < 80:
        confidence *= 0.85

    return {
        'source': source,
        'input_type': 'text',
        'content_category': best['category'],
        'confidence': round(confidence, 6),
        'top_categories': top_categories,
        'language': language,
        'language_confidence': round(float(language_conf), 6),
        'char_count': len(cleaned),
        'status': 'success',
        'processing_time_ms': round((time.time() - start) * 1000, 2),
    }


In [6]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp', '.tiff'}
TABULAR_EXTENSIONS = {'.csv', '.tsv', '.xlsx', '.xls'}
TEXT_EXTENSIONS = {'.txt', '.md', '.rtf', '.log'}
DOC_EXTENSIONS = {'.pdf', '.docx'}
CODE_EXTENSIONS = {'.py', '.js', '.ts', '.java', '.cpp', '.c', '.go', '.rs', '.html', '.css', '.xml', '.yaml', '.yml', '.sql'}
JSON_EXTENSIONS = {'.json'}


def detect_file_type(file_path: Union[str, Path]) -> str:
    path = Path(file_path)
    suffix = path.suffix.lower()

    if suffix in IMAGE_EXTENSIONS:
        return 'image'
    if suffix in TABULAR_EXTENSIONS:
        return 'tabular'
    if suffix in DOC_EXTENSIONS:
        return 'document'
    if suffix in TEXT_EXTENSIONS:
        return 'text'
    if suffix in CODE_EXTENSIONS:
        return 'code'
    if suffix in JSON_EXTENSIONS:
        return 'json'

    if HAS_FILETYPE:
        try:
            kind = filetype.guess(str(path))
            if kind is not None:
                mime = kind.mime
                if mime.startswith('image/'):
                    return 'image'
                if mime in {'application/pdf'}:
                    return 'document'
                if mime.startswith('text/'):
                    return 'text'
        except Exception:
            pass

    mime_type, _ = mimetypes.guess_type(str(path))
    if mime_type:
        if mime_type.startswith('image/'):
            return 'image'
        if mime_type == 'application/pdf':
            return 'document'
        if mime_type.startswith('text/'):
            return 'text'

    return 'unknown'


def _summarize_dataframe(df: pd.DataFrame, max_rows: int = 6, max_cols: int = 24) -> str:
    if df is None or df.empty:
        return 'Empty table.'

    cols = [str(c) for c in df.columns[:max_cols]]
    dtypes = {str(c): str(df[c].dtype) for c in df.columns[:max_cols]}

    preview = df.head(max_rows)
    preview_text = preview.to_string(index=False)

    numeric = preview.select_dtypes(include=[np.number])
    stats_text = ''
    if not numeric.empty:
        desc = numeric.describe().T.round(3)
        stats_text = desc.to_string()

    pieces = [
        f'Rows(sampled): {len(df)}',
        f'Columns: {len(df.columns)}',
        f'Column names: {cols}',
        f'Dtypes: {dtypes}',
        'Preview:',
        preview_text,
    ]
    if stats_text:
        pieces += ['Numeric summary:', stats_text]

    return '\n'.join(pieces)


def summarize_tabular_file(file_path: Union[str, Path], max_rows: int = 80) -> str:
    path = Path(file_path)
    suffix = path.suffix.lower()

    if suffix == '.csv':
        try:
            df = pd.read_csv(path, nrows=max_rows)
        except Exception:
            try:
                df = pd.read_csv(path, nrows=max_rows, sep=None, engine='python')
            except Exception:
                df = pd.read_csv(path, nrows=max_rows, encoding='latin1', on_bad_lines='skip')
    elif suffix == '.tsv':
        df = pd.read_csv(path, sep='	', nrows=max_rows)
    elif suffix in {'.xlsx', '.xls'}:
        df = pd.read_excel(path, nrows=max_rows)
    else:
        raise ValueError(f'Unsupported tabular extension: {suffix}')

    return _summarize_dataframe(df)


def _extract_pdf_text(file_path: Path, max_pages: int = 5) -> str:
    text_parts: List[str] = []
    with file_path.open('rb') as f:
        reader = PyPDF2.PdfReader(f)
        n_pages = min(max_pages, len(reader.pages))
        for i in range(n_pages):
            page_text = reader.pages[i].extract_text() or ''
            if page_text:
                text_parts.append(page_text)
    return '\n'.join(text_parts)


def _extract_docx_text(file_path: Path) -> str:
    if not HAS_DOCX or Document is None:
        raise RuntimeError('python-docx is not installed; cannot parse DOCX files.')
    doc = Document(str(file_path))
    return '\n'.join(p.text for p in doc.paragraphs if p.text and p.text.strip())


def extract_text_from_file(file_path: Union[str, Path]) -> str:
    path = Path(file_path)
    suffix = path.suffix.lower()

    if suffix in TABULAR_EXTENSIONS:
        return summarize_tabular_file(path)

    if suffix in TEXT_EXTENSIONS or suffix in CODE_EXTENSIONS:
        return path.read_text(encoding='utf-8', errors='ignore')[:MAX_TEXT_CHARS]

    if suffix == '.json':
        try:
            data = json.loads(path.read_text(encoding='utf-8', errors='ignore'))
            return json.dumps(data, ensure_ascii=False, indent=2)[:MAX_TEXT_CHARS]
        except Exception:
            return path.read_text(encoding='utf-8', errors='ignore')[:MAX_TEXT_CHARS]

    if suffix == '.pdf':
        return _extract_pdf_text(path)[:MAX_TEXT_CHARS]

    if suffix == '.docx':
        return _extract_docx_text(path)[:MAX_TEXT_CHARS]

    # Fallback: best-effort text decode
    try:
        return path.read_text(encoding='utf-8', errors='ignore')[:MAX_TEXT_CHARS]
    except Exception:
        return ''


In [7]:
def classify_image_file(file_path: Union[str, Path], top_k: int = TOP_K_DEFAULT) -> Dict[str, Any]:
    if not USE_IMAGE_MODEL or image_model is None or image_processor is None:
        return {
            'input_type': 'image',
            'content_category': 'GENERAL_OTHER',
            'confidence': 0.0,
            'top_categories': [],
            'status': 'error',
            'error': 'Image model is disabled or unavailable.',
        }

    start = time.time()
    path = Path(file_path)

    image = Image.open(path).convert('RGB')

    inputs = image_processor(
        text=CATEGORY_IMAGE_PROMPTS,
        images=image,
        return_tensors='pt',
        padding=True,
    )
    inputs = {k: v.to(image_device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = image_model(**inputs)

    if hasattr(outputs, 'logits_per_image') and outputs.logits_per_image is not None:
        logits = outputs.logits_per_image[0]
    elif hasattr(outputs, 'image_embeds') and hasattr(outputs, 'text_embeds'):
        image_embeds = outputs.image_embeds
        text_embeds = outputs.text_embeds
        image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
        text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)
        logits = (image_embeds @ text_embeds.T)[0]
    else:
        return {
            'input_type': 'image',
            'content_category': 'GENERAL_OTHER',
            'confidence': 0.0,
            'top_categories': [],
            'status': 'error',
            'error': 'Unexpected image model output format.',
        }

    probs = torch.softmax(logits, dim=-1).detach().float().cpu().numpy()
    order = np.argsort(probs)[::-1]
    top_idx = order[:max(1, top_k)]

    top_categories = [
        {
            'category': CATEGORY_LABELS[i],
            'probability': float(probs[i]),
        }
        for i in top_idx
    ]

    best = top_categories[0]
    second_prob = top_categories[1]['probability'] if len(top_categories) > 1 else 0.0
    margin = best['probability'] - second_prob
    confidence = float(min(1.0, best['probability'] + 0.25 * margin))

    return {
        'input_type': 'image',
        'content_category': best['category'],
        'confidence': round(confidence, 6),
        'top_categories': top_categories,
        'status': 'success',
        'processing_time_ms': round((time.time() - start) * 1000, 2),
    }


def classify_file(
    file_path: Union[str, Path],
    top_k: int = TOP_K_DEFAULT,
    use_reranker: bool = USE_RERANKER,
) -> Dict[str, Any]:
    start = time.time()
    path = Path(file_path).expanduser()

    if not path.exists() or not path.is_file():
        return {
            'file_path': str(path),
            'file_name': path.name,
            'status': 'error',
            'error': 'File does not exist',
            'content_category': 'GENERAL_OTHER',
            'confidence': 0.0,
            'processing_time_ms': round((time.time() - start) * 1000, 2),
        }

    file_type = detect_file_type(path)

    if file_type == 'image':
        pred = classify_image_file(path, top_k=top_k)
    else:
        extracted = extract_text_from_file(path)
        pred = classify_text_payload(
            extracted,
            source=str(path),
            top_k=top_k,
            use_reranker=use_reranker,
        )

    return {
        'file_path': str(path),
        'file_name': path.name,
        'file_type': file_type,
        'file_size_bytes': path.stat().st_size,
        'content_category': pred.get('content_category', 'GENERAL_OTHER'),
        'confidence': pred.get('confidence', 0.0),
        'top_categories': pred.get('top_categories', []),
        'language': pred.get('language'),
        'language_confidence': pred.get('language_confidence'),
        'status': pred.get('status', 'success'),
        'error': pred.get('error'),
        'processing_time_ms': round((time.time() - start) * 1000, 2),
    }


def classify_many(items: List[Union[str, Path]], assume_paths: bool = True) -> List[Dict[str, Any]]:
    results: List[Dict[str, Any]] = []
    for item in items:
        if assume_paths:
            p = Path(str(item)).expanduser()
            if p.exists() and p.is_file():
                results.append(classify_file(p))
                continue
        results.append(classify_text_payload(str(item), source='inline_text'))
    return results


def results_to_dataframe(results: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for r in results:
        top = r.get('top_categories', []) or []
        rows.append({
            'source': r.get('file_path', r.get('source', 'inline_text')),
            'file_type': r.get('file_type', r.get('input_type', 'text')),
            'category': r.get('content_category'),
            'confidence': r.get('confidence'),
            'top1': top[0]['category'] if len(top) > 0 else None,
            'top1_prob': top[0]['probability'] if len(top) > 0 else None,
            'top2': top[1]['category'] if len(top) > 1 else None,
            'top2_prob': top[1]['probability'] if len(top) > 1 else None,
            'status': r.get('status'),
            'error': r.get('error'),
            'processing_time_ms': r.get('processing_time_ms'),
        })
    return pd.DataFrame(rows)


def save_results_json(results: List[Dict[str, Any]], filename: str) -> Path:
    out_path = OUTPUT_DIR / filename
    with out_path.open('w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    return out_path


In [8]:
# Quick text-payload smoke test
sample_payloads = [
    'Q4 revenue grew 15 percent, operating margin improved, and free cash flow increased.',
    'Patient outcomes improved after treatment; diagnosis and medication adherence were tracked monthly.',
    'API latency spiked after deployment, and database connection retries increased in production logs.',
    'This note is just a casual reminder to buy groceries and call a friend this evening.',
]

text_results = [classify_text_payload(t, source=f'text_{i+1}') for i, t in enumerate(sample_payloads)]
results_to_dataframe(text_results)


,source,file_type,category,confidence,top1,top1_prob,top2,top2_prob,status,error,processing_time_ms
0,text_1,text,FINANCIAL,0.651108,FINANCIAL,0.548636,BUSINESS_COMMERCIAL,0.138748,success,None,31782.88
1,text_2,text,HEALTH_MEDICAL,0.800458,HEALTH_MEDICAL,0.652077,SCIENCE_RESEARCH,0.058552,success,None,44309.09
2,text_3,text,TECHNOLOGY_IT,0.822248,TECHNOLOGY_IT,0.667753,ENGINEERING_MANUFACTURING,0.049772,success,None,38879.12
3,text_4,text,PERSONAL_INFORMAL,0.893220,PERSONAL_INFORMAL,0.725085,GENERAL_OTHER,0.052548,success,None,39897.77


In [9]:
# File-based test using your existing Beta/test_files examples
sample_files = [
    BASE_DIR / 'test_files' / 'test_financial.txt',
    BASE_DIR / 'test_files' / 'test_medical.csv',
    BASE_DIR / 'test_files' / 'test_image.png',
    BASE_DIR / 'test_files' / 'examples' / 'ml_model.py',
]
sample_files = [p for p in sample_files if p.exists()]

file_results = [classify_file(p) for p in sample_files]
df_files = results_to_dataframe(file_results)
df_files


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/transformers/tokenization_utils_base.py:2919: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


,source,file_type,category,confidence,top1,top1_prob,top2,top2_prob,status,error,processing_time_ms
0,/Users/benjaminfalkenburg/Documents/Shannova/B...,text,FINANCIAL,0.682333,FINANCIAL,0.565147,BUSINESS_COMMERCIAL,0.096403,success,None,53291.51
1,/Users/benjaminfalkenburg/Documents/Shannova/B...,tabular,HEALTH_MEDICAL,0.782842,HEALTH_MEDICAL,0.638287,SCIENCE_RESEARCH,0.060066,success,None,45538.94
2,/Users/benjaminfalkenburg/Documents/Shannova/B...,image,TECHNOLOGY_IT,0.328216,TECHNOLOGY_IT,0.298613,CREATIVE_MEDIA,0.180200,success,None,4633.13
3,/Users/benjaminfalkenburg/Documents/Shannova/B...,code,EDUCATION_ACADEMIC,0.192996,EDUCATION_ACADEMIC,0.186196,SCIENCE_RESEARCH,0.158996,success,None,54199.50


In [10]:
# Optional: persist latest test results
from datetime import datetime

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
out_json = save_results_json(file_results, f'nb2_demo_results_{stamp}.json')
print('Saved:', out_json)


Saved: /Users/benjaminfalkenburg/Documents/Shannova/Beta/outputs/nb2_demo_results_20260203_022151.json


## How to use this pipeline

- Classify one file:
```python
result = classify_file('/path/to/file.csv')
result['content_category'], result['confidence']
```

- Classify one inline data piece:
```python
result = classify_text_payload('your text payload here')
```

- Classify many inputs:
```python
batch = classify_many([
    '/path/to/file1.pdf',
    '/path/to/file2.png',
    'inline text payload',
], assume_paths=True)
results_to_dataframe(batch)
```

## Tuning for your exact definition of "accurate"
1. Edit `CATEGORY_DEFINITIONS` descriptions/keywords/examples.
2. Keep `USE_RERANKER=True` for harder boundary cases.
3. If image classification is not needed, set `USE_IMAGE_MODEL=False` for faster startup.
4. Re-run the model-loading cell after changing model/category settings.


In [15]:
import math

def get_stirling_signed(n, k):
    """Calculates actual s(n, k) using recurrence."""
    if n == k: return 1
    if k <= 0 or k > n: return 0
    memo = {}
    def s(n, k):
        if (n, k) in memo: return memo[(n, k)]
        if n == k: return 1
        if k <= 0 or k > n: return 0
        res = s(n-1, k-1) - (n-1) * s(n-1, k)
        memo[(n, k)] = res
        return res
    return s(n, k)

def nCr(n, r):
    """Helper for binomial coefficients."""
    return math.comb(n, r) if 0 <= r <= n else 0

def calculate_head_coefficients(n):
    """Calculates C0 through C5 using the binomial formulas."""
    c0 = 1
    c1 = -nCr(n, 2)
    c2 = 3*nCr(n, 4) + 2*nCr(n, 3)
    c3 = -(15*nCr(n, 6) + 20*nCr(n, 5) + 6*nCr(n, 4))
    c4 = 105*nCr(n, 8) + 210*nCr(n, 7) + 130*nCr(n, 6) + 24*nCr(n, 5)
    c5 = -(945*nCr(n, 10) + 2520*nCr(n, 9) + 2380*nCr(n, 8) + 924*nCr(n, 7) + 120*nCr(n, 6))
    return [c0, c1, c2, c3, c4, c5]

# Example test for n=20
n_val = 20
head = calculate_head_coefficients(n_val)
for i, val in enumerate(head):
    actual = get_stirling_signed(n_val, n_val - i)
    print(f"C{i} (n={n_val}): {val} | Match: {val == actual}")

C0 (n=20): 1 | Match: True
C1 (n=20): -190 | Match: True
C2 (n=20): 16815 | Match: True
C3 (n=20): -920550 | Match: True
C4 (n=20): 34916946 | Match: True
C5 (n=20): -973941900 | Match: True


In [ ]:
import math

def stirling_first(n, k):
    """Computes signed Stirling numbers of the first kind s(n, k)."""
    if n == k: return 1
    if k == 0: return 0
    # Recurrence: s(n, k) = s(n-1, k-1) - (n-1)*s(n-1, k)
    dp = [[0] * (k + 1) for _ in range(n + 1)]
    dp[0][0] = 1
    for i in range(1, n + 1):
        for j in range(1, min(i, k) + 1):
            dp[i][j] = dp[i-1][j-1] - (i-1) * dp[i-1][j]
    return dp[n][k]

def compute_head(n, j):
    """
    Computes H_j(n) = sum_{i=0}^{j-1} s(n, n-i) * n^(n-i)
    j is the number of terms.
    """
    total = 0
    # Sum from i=0 to j-1. The power is k = n-i.
    for i in range(j):
        power = n - i
        coeff = stirling_first(n, power)
        term = coeff * (n ** power)
        total += term
    return total

def is_quadratic_residue(val, m):
    """Checks if val is a quadratic residue modulo m."""
    val = val % m
    # For optimization in larger apps, use Legendre symbol. 
    # For this scale, brute force search is sufficient and explicit.
    for i in range(m):
        if (i * i) % m == val:
            return True
    return False

def analyze_brocard_structure():
    print("--- 1. Verifying 'Zero Identity' ---")
    print("Checking sum_{k=2}^n s(n,k)n^k (excluding the n! term at k=1)")
    
    for n in range(4, 25):
        partial_sum = 0
        for k in range(2, n + 1):
            partial_sum += stirling_first(n, k) * (n ** k)
        
        factorial_n = math.factorial(n)
        
        # Determine the structure
        status = ""
        if n % 2 != 0:
            if partial_sum == 0:
                status = "CONFIRMED Zero Identity (Sum = 0)"
            else:
                status = f"FAILED (Sum = {partial_sum})"
        else:
            if partial_sum == 2 * factorial_n:
                status = f"Even n behavior (Sum = 2n!)"
            else:
                status = f"Unknown behavior (Sum = {partial_sum})"
                
        print(f"n={n}: {status}")

    print("\n--- 2. Modular Constraints for Brocard's Problem ---")
    print("Condition: n! + 1 = m^2 => R_j(n) + 1 must be a Quadratic Residue mod n^(n-j+1)")
    
    # Test for known solutions (4, 5, 7) and non-solutions (6, 8, 9)
    test_n = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25] 
    
    print(f"{'n':<4} {'j':<4} {'Is_Sol':<8} {'Modulus':<10} {'Residue':<12} {'Is_QR':<6}")
    print("-" * 65)
    
    for n in test_n:
        n_fact = math.factorial(n)
        is_brocard_sol = (math.isqrt(n_fact + 1) ** 2 == n_fact + 1)
            
        for j in range(2, 5): # Truncation depth
            if j >= n: continue
            
            # Since H_j(n) is divisible by M = n^(n-j+1), 
            # R_j(n) = n! - H_j(n) == n! (mod M)
            # We are essentially checking if n! + 1 is a QR mod M.
            
            mod_power = n - j + 1
            modulus = n ** mod_power
            
            val_to_check = n_fact + 1
            residue = val_to_check % modulus
            is_qr = is_quadratic_residue(residue, modulus)
            
            print(f"{n:<4} {j:<4} {str(is_brocard_sol):<8} n^{mod_power:<8} {residue:<12} {str(is_qr):<6}")

if __name__ == "__main__":
    analyze_brocard_structure()

--- 1. Verifying 'Zero Identity' ---
Checking sum_{k=2}^n s(n,k)n^k (excluding the n! term at k=1)
n=4: Even n behavior (Sum = 2n!)
n=5: CONFIRMED Zero Identity (Sum = 0)
n=6: Even n behavior (Sum = 2n!)
n=7: CONFIRMED Zero Identity (Sum = 0)
n=8: Even n behavior (Sum = 2n!)
n=9: CONFIRMED Zero Identity (Sum = 0)
n=10: Even n behavior (Sum = 2n!)
n=11: CONFIRMED Zero Identity (Sum = 0)
n=12: Even n behavior (Sum = 2n!)
n=13: CONFIRMED Zero Identity (Sum = 0)
n=14: Even n behavior (Sum = 2n!)
n=15: CONFIRMED Zero Identity (Sum = 0)
n=16: Even n behavior (Sum = 2n!)
n=17: CONFIRMED Zero Identity (Sum = 0)
n=18: Even n behavior (Sum = 2n!)
n=19: CONFIRMED Zero Identity (Sum = 0)
n=20: Even n behavior (Sum = 2n!)
n=21: CONFIRMED Zero Identity (Sum = 0)
n=22: Even n behavior (Sum = 2n!)
n=23: CONFIRMED Zero Identity (Sum = 0)
n=24: Even n behavior (Sum = 2n!)

--- 2. Modular Constraints for Brocard's Problem ---
Condition: n! + 1 = m^2 => R_j(n) + 1 must be a Quadratic Residue mod n^(n-j+1)

In [ ]:
import math

def stirling_first(n, k):
    """Computes signed Stirling numbers of the first kind s(n, k)."""
    if n == k: return 1
    if k == 0: return 0
    # Recurrence: s(n, k) = s(n-1, k-1) - (n-1)*s(n-1, k)
    dp = [[0] * (k + 1) for _ in range(n + 1)]
    dp[0][0] = 1
    for i in range(1, n + 1):
        for j in range(1, min(i, k) + 1):
            dp[i][j] = dp[i-1][j-1] - (i-1) * dp[i-1][j]
    return dp[n][k]

def compute_head(n, j):
    """
    Computes H_j(n) = sum_{i=0}^{j-1} s(n, n-i) * n^(n-i)
    j is the number of terms.
    """
    total = 0
    # Sum from i=0 to j-1. The power is k = n-i.
    for i in range(j):
        power = n - i
        coeff = stirling_first(n, power)
        term = coeff * (n ** power)
        total += term
    return total

def is_quadratic_residue(val, m):
    """Checks if val is a quadratic residue modulo m."""
    val = val % m
    # For optimization in larger apps, use Legendre symbol. 
    # For this scale, brute force search is sufficient and explicit.
    for i in range(m):
        if (i * i) % m == val:
            return True
    return False

def analyze_brocard_structure():
    print("--- 1. Verifying 'Zero Identity' ---")
    print("Checking sum_{k=2}^n s(n,k)n^k (excluding the n! term at k=1)")
    
    for n in range(4, 25):
        partial_sum = 0
        for k in range(2, n + 1):
            partial_sum += stirling_first(n, k) * (n ** k)
        
        factorial_n = math.factorial(n)
        
        # Determine the structure
        status = ""
        if n % 2 != 0:
            if partial_sum == 0:
                status = "CONFIRMED Zero Identity (Sum = 0)"
            else:
                status = f"FAILED (Sum = {partial_sum})"
        else:
            if partial_sum == 2 * factorial_n:
                status = f"Even n behavior (Sum = 2n!)"
            else:
                status = f"Unknown behavior (Sum = {partial_sum})"
                
        print(f"n={n}: {status}")

    print("\n--- 2. Modular Constraints for Brocard's Problem ---")
    print("Condition: n! + 1 = m^2 => R_j(n) + 1 must be a Quadratic Residue mod n^(n-j+1)")
    
    # Test for known solutions (4, 5, 7) and non-solutions (6, 8, 9)
    test_n = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25] 
    
    print(f"{'n':<4} {'j':<4} {'Is_Sol':<8} {'Modulus':<10} {'Residue':<12} {'Is_QR':<6}")
    print("-" * 65)
    
    for n in test_n:
        n_fact = math.factorial(n)
        is_brocard_sol = (math.isqrt(n_fact + 1) ** 2 == n_fact + 1)
            
        for j in range(2, 5): # Truncation depth
            if j >= n: continue
            
            # Since H_j(n) is divisible by M = n^(n-j+1), 
            # R_j(n) = n! - H_j(n) == n! (mod M)
            # We are essentially checking if n! + 1 is a QR mod M.
            
            mod_power = n - j + 1
            modulus = n ** mod_power
            
            val_to_check = n_fact + 1
            residue = val_to_check % modulus
            is_qr = is_quadratic_residue(residue, modulus)
            
            print(f"{n:<4} {j:<4} {str(is_brocard_sol):<8} n^{mod_power:<8} {residue:<12} {str(is_qr):<6}")

if __name__ == "__main__":
    analyze_brocard_structure()

--- 1. Verifying 'Zero Identity' ---
Checking sum_{k=2}^n s(n,k)n^k (excluding the n! term at k=1)
n=4: Even n behavior (Sum = 2n!)
n=5: CONFIRMED Zero Identity (Sum = 0)
n=6: Even n behavior (Sum = 2n!)
n=7: CONFIRMED Zero Identity (Sum = 0)
n=8: Even n behavior (Sum = 2n!)
n=9: CONFIRMED Zero Identity (Sum = 0)
n=10: Even n behavior (Sum = 2n!)
n=11: CONFIRMED Zero Identity (Sum = 0)
n=12: Even n behavior (Sum = 2n!)
n=13: CONFIRMED Zero Identity (Sum = 0)
n=14: Even n behavior (Sum = 2n!)
n=15: CONFIRMED Zero Identity (Sum = 0)
n=16: Even n behavior (Sum = 2n!)
n=17: CONFIRMED Zero Identity (Sum = 0)
n=18: Even n behavior (Sum = 2n!)
n=19: CONFIRMED Zero Identity (Sum = 0)
n=20: Even n behavior (Sum = 2n!)
n=21: CONFIRMED Zero Identity (Sum = 0)
n=22: Even n behavior (Sum = 2n!)
n=23: CONFIRMED Zero Identity (Sum = 0)
n=24: Even n behavior (Sum = 2n!)

--- 2. Modular Constraints for Brocard's Problem ---
Condition: n! + 1 = m^2 => R_j(n) + 1 must be a Quadratic Residue mod n^(n-j+1)